# 📊 Institutional Risk Management Dashboard

### Value-at-Risk (VaR), Expected Shortfall (CVaR), Stress Testing & Portfolio Analytics

---

## Objective

This notebook builds a professional portfolio risk management system similar to those used by:

- Hedge Funds
- Investment Banks
- Asset Managers
- Pension Funds
- Risk Management Teams

The dashboard will:

- Download market data
- Build a portfolio
- Calculate returns
- Estimate portfolio risk
- Compute Historical VaR
- Compute Parametric VaR
- Compute Monte Carlo VaR
- Compute Expected Shortfall
- Perform stress testing
- Visualize portfolio analytics

In [5]:
# ==========================================================
# INSTALL REQUIRED PACKAGES
# ==========================================================

!pip install -q yfinance plotly scipy fredapi

In [6]:
# ==========================================================
# IMPORT LIBRARIES
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import logging
from datetime import datetime
from typing import List, Dict

import numpy as np
import pandas as pd

import yfinance as yf

from scipy.stats import norm

import plotly.graph_objects as go
import plotly.express as px

from plotly.subplots import make_subplots

In [7]:
# ==========================================================
# LOGGING CONFIGURATION
# ==========================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger(__name__)

In [8]:
# ==========================================================
# PROJECT CONFIGURATION
# ==========================================================

TICKERS = [
    "AAPL",
    "MSFT",
    "GOOGL",
    "AMZN",
    "NVDA"
]

START_DATE = "2019-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")

INITIAL_PORTFOLIO_VALUE = 1_000_000

CONFIDENCE_LEVEL = 0.95

TRADING_DAYS = 252

In [9]:
# ==========================================================
# DOWNLOAD MARKET DATA
# ==========================================================

def download_prices(
    tickers: List[str],
    start: str,
    end: str
) -> pd.DataFrame:
    """
    Downloads adjusted closing prices from Yahoo Finance.

    Parameters
    ----------
    tickers : list
        List of stock tickers.

    start : str

    end : str

    Returns
    -------
    pd.DataFrame
    """

    try:

        logger.info("Downloading market data...")

        prices = yf.download(
            tickers,
            start=start,
            end=end,
            auto_adjust=True,
            progress=False
        )["Close"]

        logger.info("Download successful.")

        return prices

    except Exception as e:

        logger.error(f"Download failed: {e}")

        raise

In [10]:
prices = download_prices(
    TICKERS,
    START_DATE,
    END_DATE
)

prices.head()

Ticker,AAPL,AMZN,GOOGL,MSFT,NVDA
Date,,,,,
2019-01-02,37.469200,76.956497,52.270508,94.193138,3.373052
2019-01-03,33.737003,75.014000,50.822842,90.727989,3.169263
2019-01-04,35.177200,78.769501,53.429726,94.947647,3.372310
2019-01-07,35.098907,81.475502,53.323174,95.068741,3.550842
2019-01-08,35.768005,82.829002,53.791523,95.758057,3.462443


In [11]:
# ==========================================================
# DATA VALIDATION
# ==========================================================

def validate_prices(df: pd.DataFrame):

    if df.empty:
        raise ValueError("Dataset is empty.")

    if df.isnull().all().all():
        raise ValueError("All observations are missing.")

    logger.info(f"Rows: {len(df)}")

    logger.info(f"Columns: {len(df.columns)}")

    logger.info("Validation completed.")

    validate_prices(prices)

In [12]:
# ==========================================================
# CLEAN DATA
# ==========================================================

prices = prices.sort_index()

prices = prices.ffill()

prices = prices.dropna()

prices.head()

Ticker,AAPL,AMZN,GOOGL,MSFT,NVDA
Date,,,,,
2019-01-02,37.469200,76.956497,52.270508,94.193138,3.373052
2019-01-03,33.737003,75.014000,50.822842,90.727989,3.169263
2019-01-04,35.177200,78.769501,53.429726,94.947647,3.372310
2019-01-07,35.098907,81.475502,53.323174,95.068741,3.550842
2019-01-08,35.768005,82.829002,53.791523,95.758057,3.462443


In [13]:
missing = prices.isna().sum()

missing

,0
Ticker,
AAPL,0
AMZN,0
GOOGL,0
MSFT,0
NVDA,0


In [14]:
prices.describe().T

,count,mean,std,min,25%,50%,75%,max
Ticker,,,,,,,,
AAPL,1898.0,157.636877,68.121851,33.737003,117.126740,157.093887,206.275005,333.739990
AMZN,1898.0,155.016258,48.743054,75.014000,109.582499,157.544502,186.122505,274.989990
GOOGL,1898.0,138.452614,76.379950,50.822842,85.685333,124.073559,163.408836,402.379669
MSFT,1898.0,296.019304,115.551136,90.727989,205.870430,283.725113,400.956566,538.658569
NVDA,1898.0,60.477678,65.660332,3.169263,12.881893,23.280354,115.818535,235.465576


In [15]:
# ==========================================================
# PORTFOLIO WEIGHTS
# ==========================================================

weights = np.repeat(
    1/len(TICKERS),
    len(TICKERS)
)

weights

array([0.2, 0.2, 0.2, 0.2, 0.2])

In [16]:
# ==========================================================
# DAILY RETURNS
# ==========================================================

returns = prices.pct_change().dropna()

returns.head()

Ticker,AAPL,AMZN,GOOGL,MSFT,NVDA
Date,,,,,
2019-01-03,-0.099607,-0.025241,-0.027696,-0.036788,-0.060417
2019-01-04,0.042689,0.050064,0.051294,0.046509,0.064068
2019-01-07,-0.002226,0.034353,-0.001994,0.001275,0.052941
2019-01-08,0.019063,0.016612,0.008783,0.007251,-0.024895
2019-01-09,0.016982,0.001714,-0.003428,0.014300,0.019666


In [17]:
portfolio_returns = returns @ weights

portfolio_returns.head()

,0
Date,
2019-01-03,-0.049950
2019-01-04,0.050925
2019-01-07,0.016870
2019-01-08,0.005363
2019-01-09,0.009847


In [18]:
portfolio_value = (
    1 + portfolio_returns
).cumprod() * INITIAL_PORTFOLIO_VALUE

portfolio_value.head()

,0
Date,
2019-01-03,9.500503e+05
2019-01-04,9.984312e+05
2019-01-07,1.015275e+06
2019-01-08,1.020719e+06
2019-01-09,1.030770e+06


In [19]:
fig = go.Figure()

fig.add_trace(

    go.Scatter(

        x=portfolio_value.index,

        y=portfolio_value,

        mode="lines",

        name="Portfolio"

    )

)

fig.update_layout(

    title="Portfolio Value",

    template="plotly_white",

    height=600

)

fig.show()

In [20]:
corr = returns.corr()

fig = px.imshow(

    corr,

    text_auto=True,

    color_continuous_scale="RdBu_r",

    title="Correlation Matrix"

)

fig.show()

In [21]:
cov = returns.cov()

fig = px.imshow(

    cov,

    text_auto=True,

    color_continuous_scale="Viridis",

    title="Covariance Matrix"

)

fig.show()

In [22]:
summary = pd.DataFrame({

    "Annual Return":
        returns.mean()*252,

    "Annual Volatility":
        returns.std()*np.sqrt(252)

})

summary

,Annual Return,Annual Volatility
Ticker,,
AAPL,0.334705,0.307573
AMZN,0.211232,0.339187
GOOGL,0.298798,0.313669
MSFT,0.230317,0.287700
NVDA,0.678431,0.506993


# 📉 Risk Analytics

In this section we implement several industry-standard risk measures:

- Historical Value-at-Risk (VaR)
- Parametric (Variance-Covariance) VaR
- Monte Carlo VaR
- Expected Shortfall (CVaR)

These metrics estimate the potential downside risk of the portfolio.

In [23]:
# ==========================================================
# PORTFOLIO STATISTICS
# ==========================================================

portfolio_mean = portfolio_returns.mean()
portfolio_std = portfolio_returns.std()

print(f"Average Daily Return : {portfolio_mean:.6f}")
print(f"Daily Volatility     : {portfolio_std:.6f}")

Average Daily Return : 0.001392
Daily Volatility     : 0.018262


In [24]:
# ==========================================================
# HISTORICAL VAR
# ==========================================================

def historical_var(
    returns: pd.Series,
    confidence: float = 0.95
) -> float:
    """
    Calculate Historical Value-at-Risk.

    Parameters
    ----------
    returns : pd.Series
        Portfolio returns.

    confidence : float
        Confidence level (default 95%).

    Returns
    -------
    float
        Historical VaR
    """

    percentile = (1 - confidence) * 100

    return np.percentile(returns, percentile)

In [25]:
hist_var_95 = historical_var(portfolio_returns, 0.95)
hist_var_99 = historical_var(portfolio_returns, 0.99)

print(f"95% Historical VaR : {hist_var_95:.2%}")
print(f"99% Historical VaR : {hist_var_99:.2%}")

95% Historical VaR : -2.95%
99% Historical VaR : -4.85%


In [26]:
# ==========================================================
# PARAMETRIC VAR
# ==========================================================

def parametric_var(
    mean: float,
    std: float,
    confidence: float = 0.95
) -> float:
    """
    Variance-Covariance VaR assuming normally distributed returns.
    """

    z = norm.ppf(1 - confidence)

    return mean + z * std

In [27]:
param_var_95 = parametric_var(
    portfolio_mean,
    portfolio_std,
    0.95
)

param_var_99 = parametric_var(
    portfolio_mean,
    portfolio_std,
    0.99
)

print(f"95% Parametric VaR : {param_var_95:.2%}")
print(f"99% Parametric VaR : {param_var_99:.2%}")

95% Parametric VaR : -2.86%
99% Parametric VaR : -4.11%


In [28]:
# ==========================================================
# EXPECTED SHORTFALL
# ==========================================================

def expected_shortfall(
    returns: pd.Series,
    confidence: float = 0.95
) -> float:

    var = historical_var(returns, confidence)

    losses = returns[returns <= var]

    return losses.mean()

In [29]:
es95 = expected_shortfall(
    portfolio_returns,
    0.95
)

es99 = expected_shortfall(
    portfolio_returns,
    0.99
)

print(f"95% Expected Shortfall : {es95:.2%}")
print(f"99% Expected Shortfall : {es99:.2%}")

95% Expected Shortfall : -4.15%
99% Expected Shortfall : -6.22%


In [30]:
# ==========================================================
# MONTE CARLO VAR
# ==========================================================

np.random.seed(42)

num_simulations = 100000

simulated_returns = np.random.normal(
    portfolio_mean,
    portfolio_std,
    num_simulations
)

mc_var95 = np.percentile(simulated_returns, 5)
mc_var99 = np.percentile(simulated_returns, 1)

print(f"95% Monte Carlo VaR : {mc_var95:.2%}")
print(f"99% Monte Carlo VaR : {mc_var99:.2%}")

95% Monte Carlo VaR : -2.86%
99% Monte Carlo VaR : -4.12%


In [31]:
fig = px.histogram(
    simulated_returns,
    nbins=100,
    title="Monte Carlo Return Distribution"
)

fig.add_vline(
    x=mc_var95,
    line_color="red",
    annotation_text="95% VaR"
)

fig.add_vline(
    x=es95,
    line_color="orange",
    annotation_text="Expected Shortfall"
)

fig.update_layout(template="plotly_white")

fig.show()

In [32]:
rolling_var = portfolio_returns.rolling(252).apply(
    lambda x: historical_var(pd.Series(x), 0.95)
)

In [33]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=rolling_var.index,
        y=rolling_var,
        name="Rolling 95% VaR"
    )
)

fig.update_layout(
    title="Rolling Historical VaR",
    template="plotly_white",
    height=550
)

fig.show()

In [34]:
risk_summary = pd.DataFrame({
    "Metric": [
        "Historical VaR (95%)",
        "Historical VaR (99%)",
        "Parametric VaR (95%)",
        "Parametric VaR (99%)",
        "Expected Shortfall (95%)",
        "Expected Shortfall (99%)",
        "Monte Carlo VaR (95%)",
        "Monte Carlo VaR (99%)"
    ],
    "Value": [
        hist_var_95,
        hist_var_99,
        param_var_95,
        param_var_99,
        es95,
        es99,
        mc_var95,
        mc_var99
    ]
})

risk_summary["Value"] = risk_summary["Value"].map(lambda x: f"{x:.2%}")

risk_summary

,Metric,Value
0,Historical VaR (95%),-2.95%
1,Historical VaR (99%),-4.85%
2,Parametric VaR (95%),-2.86%
3,Parametric VaR (99%),-4.11%
4,Expected Shortfall (95%),-4.15%
5,Expected Shortfall (99%),-6.22%
6,Monte Carlo VaR (95%),-2.86%
7,Monte Carlo VaR (99%),-4.12%


# 📊 Portfolio Analytics Dashboard

This section evaluates portfolio performance using institutional portfolio analytics.

Metrics include:

- Annual Return
- Annual Volatility
- Sharpe Ratio
- Maximum Drawdown
- Rolling Volatility
- Rolling Sharpe Ratio
- Drawdown Analysis

In [35]:
# ==========================================================
# PERFORMANCE METRICS
# ==========================================================

RISK_FREE_RATE = 0.02  # 2% annually

annual_return = portfolio_returns.mean() * TRADING_DAYS

annual_volatility = portfolio_returns.std() * np.sqrt(TRADING_DAYS)

sharpe_ratio = (
    annual_return - RISK_FREE_RATE
) / annual_volatility

print(f"Annual Return     : {annual_return:.2%}")
print(f"Annual Volatility : {annual_volatility:.2%}")
print(f"Sharpe Ratio      : {sharpe_ratio:.2f}")

Annual Return     : 35.07%
Annual Volatility : 28.99%
Sharpe Ratio      : 1.14


In [36]:
# ==========================================================
# ROLLING VOLATILITY
# ==========================================================

rolling_volatility = (
    portfolio_returns
    .rolling(21)
    .std()
    * np.sqrt(TRADING_DAYS)
)

In [37]:
fig = go.Figure()

fig.add_trace(

    go.Scatter(

        x=rolling_volatility.index,

        y=rolling_volatility,

        name="21-Day Rolling Volatility"

    )

)

fig.update_layout(

    template="plotly_white",

    title="Rolling Annualized Volatility",

    height=550

)

fig.show()

In [38]:
# ==========================================================
# DRAWDOWN
# ==========================================================

running_max = portfolio_value.cummax()

drawdown = (
    portfolio_value - running_max
) / running_max

In [39]:
max_drawdown = drawdown.min()

print(f"Maximum Drawdown : {max_drawdown:.2%}")

Maximum Drawdown : -41.83%


In [40]:
fig = go.Figure()

fig.add_trace(

    go.Scatter(

        x=drawdown.index,

        y=drawdown,

        fill="tozeroy",

        name="Drawdown"

    )

)

fig.update_layout(

    template="plotly_white",

    title="Portfolio Drawdown",

    height=550

)

fig.show()

In [41]:
fig = px.histogram(

    portfolio_returns,

    nbins=80,

    title="Daily Return Distribution"

)

fig.update_layout(

    template="plotly_white"

)

fig.show()

In [42]:
rolling_return = (

    portfolio_returns

    .rolling(252)

    .mean()

    * TRADING_DAYS

)

rolling_std = (

    portfolio_returns

    .rolling(252)

    .std()

    * np.sqrt(TRADING_DAYS)

)

rolling_sharpe = (

    rolling_return - RISK_FREE_RATE

) / rolling_std

In [43]:
fig = go.Figure()

fig.add_trace(

    go.Scatter(

        x=rolling_sharpe.index,

        y=rolling_sharpe,

        name="Rolling Sharpe"

    )

)

fig.update_layout(

    title="Rolling Sharpe Ratio",

    template="plotly_white",

    height=550

)

fig.show()

In [44]:
dashboard = pd.DataFrame({

    "Metric":[

        "Annual Return",

        "Annual Volatility",

        "Sharpe Ratio",

        "Maximum Drawdown",

        "Historical VaR (95%)",

        "Expected Shortfall",

        "Monte Carlo VaR"

    ],

    "Value":[

        f"{annual_return:.2%}",

        f"{annual_volatility:.2%}",

        f"{sharpe_ratio:.2f}",

        f"{max_drawdown:.2%}",

        f"{hist_var_95:.2%}",

        f"{es95:.2%}",

        f"{mc_var95:.2%}"

    ]

})

dashboard

,Metric,Value
0,Annual Return,35.07%
1,Annual Volatility,28.99%
2,Sharpe Ratio,1.14
3,Maximum Drawdown,-41.83%
4,Historical VaR (95%),-2.95%
5,Expected Shortfall,-4.15%
6,Monte Carlo VaR,-2.86%


In [45]:
fig = go.Figure()

fig.add_trace(

    go.Indicator(

        mode="number",

        value=annual_return*100,

        title={"text":"Annual Return (%)"}

    )

)

fig.update_layout(

    height=250,

    template="plotly_white"

)

fig.show()